<a href="https://colab.research.google.com/github/sumaiiiyyah/code/blob/master/Ligand_Derivatives.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q rdkit pubchempy pandas requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 33.5 MB/s eta 0:00:00


In [2]:
# ============================================================
# CELL 2 — FINAL 24 LIGANDS
# ============================================================

ligand_cids = {

    # SATURATED NITROGEN HETEROCYCLES
    "Piperazine": 4837,
    "Morphine": 5288826,
    "Azepane": 8119,
    "Azetidine": 10422,

    # AROMATIC NITROGEN HETEROCYCLES
    "Imidazole": 795,
    "Pyrazine": 9261,
    "Quinoline": 7047,

    # OXYGEN-CONTAINING HETEROCYCLES
    "Tetrahydrofuran": 8028,
    "Tetrahydropyran": 8894,
    "Dioxolane": 12586,
    "Dioxane": 31275,

    # SULFUR-CONTAINING HETEROCYCLES
    "Thiophene": 8030,
    "Thiazole": 9256,
    "Thiomorpholine": 67164,

    # FUSED RING SYSTEMS
    "Benzimidazole": 5798,
    "Benzoxazole": 9228,
    "Benzothiazole": 7222,
    "Indazole": 9221,
    "Isoquinoline": 8405,

    # OTHERS
    "Oxazole": 9255,
    "Oxadiazole": 10197612,
    "Triazole": 9257,
    "Tetrazole": 67519,

    # INDOLE — AS INSTRUCTED
    "Indole": 1981
}

print("Number of parent ligands:", len(ligand_cids))

for i, (ligand, cid) in enumerate(
    ligand_cids.items(),
    start=1
):
    print(f"{i:02d}. {ligand} -> CID {cid}")

Number of parent ligands: 24
01. Piperazine -> CID 4837
02. Morphine -> CID 5288826
03. Azepane -> CID 8119
04. Azetidine -> CID 10422
05. Imidazole -> CID 795
06. Pyrazine -> CID 9261
07. Quinoline -> CID 7047
08. Tetrahydrofuran -> CID 8028
09. Tetrahydropyran -> CID 8894
10. Dioxolane -> CID 12586
11. Dioxane -> CID 31275
12. Thiophene -> CID 8030
13. Thiazole -> CID 9256
14. Thiomorpholine -> CID 67164
15. Benzimidazole -> CID 5798
16. Benzoxazole -> CID 9228
17. Benzothiazole -> CID 7222
18. Indazole -> CID 9221
19. Isoquinoline -> CID 8405
20. Oxazole -> CID 9255
21. Oxadiazole -> CID 10197612
22. Triazole -> CID 9257
23. Tetrazole -> CID 67519
24. Indole -> CID 1981


In [3]:
# ============================================================
# FINAL PROJECT FOLDERS
# ============================================================

from pathlib import Path

PROJECT_DIR = Path(
    "/content/ICM_Ligand_Derivative_Project_FINAL"
)

LIGAND_DIR = PROJECT_DIR / "01_Ligands"
PROTEIN_DIR = PROJECT_DIR / "02_Proteins"
SUMMARY_DIR = PROJECT_DIR / "03_Summary"

PROJECT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

LIGAND_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PROTEIN_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SUMMARY_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# One folder per ligand
for number, ligand in enumerate(
    ligand_cids.keys(),
    start=1
):

    ligand_folder = (
        LIGAND_DIR /
        f"{number:02d}_{ligand}"
    )

    (ligand_folder / "raw").mkdir(
        parents=True,
        exist_ok=True
    )

    (ligand_folder / "QC_clean").mkdir(
        parents=True,
        exist_ok=True
    )

print("Folder structure created.")
print(PROJECT_DIR)

Folder structure created.
/content/ICM_Ligand_Derivative_Project_FINAL


In [4]:
# ============================================================
# IMPORTS
# ============================================================

import time
import requests
import pandas as pd

from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import SDWriter
from rdkit.Chem import rdForceFieldHelpers

print("All libraries loaded.")

All libraries loaded.


In [5]:
# ============================================================
# PUBCHEM 2D SIMILARITY SEARCH
# ============================================================

PUBCHEM_BASE = (
    "https://pubchem.ncbi.nlm.nih.gov/rest/pug"
)


def find_similar_cids(
    cid,
    threshold=90,
    max_records=500
):

    url = (
        f"{PUBCHEM_BASE}/compound/"
        f"fastsimilarity_2d/"
        f"cid/{cid}/cids/JSON"
    )

    params = {
        "Threshold": threshold,
        "MaxRecords": max_records
    }

    for attempt in range(3):

        try:

            response = requests.get(
                url,
                params=params,
                timeout=120
            )

            # Successful response
            if response.status_code == 200:

                data = response.json()

                cids = (
                    data
                    .get("IdentifierList", {})
                    .get("CID", [])
                )

                return [
                    int(x)
                    for x in cids
                ]

            # Temporary server problem
            if response.status_code in [429, 500, 502, 503, 504]:

                print(
                    f"PubChem temporary error "
                    f"{response.status_code}. "
                    f"Retry {attempt + 1}/3..."
                )

                time.sleep(
                    5 * (attempt + 1)
                )

                continue

            print(
                "PubChem search failed:",
                response.status_code
            )

            print(
                response.text[:500]
            )

            return []

        except Exception as e:

            print(
                f"Request error: {e}"
            )

            time.sleep(
                5 * (attempt + 1)
            )

    return []

In [6]:
# ============================================================
# PUBCHEM COMPOUND INFORMATION
# ============================================================

def get_properties_for_cids(
    cids,
    batch_size=100
):

    all_records = []

    for start in range(
        0,
        len(cids),
        batch_size
    ):

        batch = cids[
            start:start + batch_size
        ]

        cid_string = ",".join(
            str(x) for x in batch
        )

        url = (
            f"{PUBCHEM_BASE}/compound/"
            f"cid/{cid_string}/property/"
            f"Title,"
            f"IUPACName,"
            f"CanonicalSMILES,"
            f"IsomericSMILES,"
            f"MolecularFormula,"
            f"MolecularWeight,"
            f"HeavyAtomCount/"
            f"JSON"
        )

        for attempt in range(3):

            try:

                response = requests.get(
                    url,
                    timeout=120
                )

                if response.status_code == 200:

                    data = response.json()

                    records = (
                        data
                        .get(
                            "PropertyTable",
                            {}
                        )
                        .get(
                            "Properties",
                            []
                        )
                    )

                    all_records.extend(
                        records
                    )

                    break

                elif response.status_code in [
                    429,
                    500,
                    502,
                    503,
                    504
                ]:

                    print(
                        f"Property request "
                        f"temporary error "
                        f"{response.status_code}"
                    )

                    time.sleep(
                        5 * (attempt + 1)
                    )

                else:

                    print(
                        "Property request failed:",
                        response.status_code
                    )

                    break

            except Exception as e:

                print(
                    "Property request error:",
                    e
                )

                time.sleep(
                    5 * (attempt + 1)
                )

        # Stay below PubChem's request-rate guidance
        time.sleep(0.5)

    return all_records

In [7]:
# ============================================================
# METAL FILTER
# ============================================================

METALS = {
    "Li", "Na", "K", "Rb", "Cs",
    "Be", "Mg", "Ca", "Sr", "Ba",

    "Al", "Ga", "In", "Tl",

    "Sc", "Y", "La",

    "Ti", "Zr", "Hf",

    "V", "Nb", "Ta",

    "Cr", "Mo", "W",

    "Mn", "Tc", "Re",

    "Fe", "Ru", "Os",

    "Co", "Rh", "Ir",

    "Ni", "Pd", "Pt",

    "Cu", "Ag", "Au",

    "Zn", "Cd", "Hg",

    "Gd", "Tb", "Dy",
    "Ho", "Er", "Tm",
    "Yb", "Lu",

    "U", "Th"
}


def contains_metal(mol):

    for atom in mol.GetAtoms():

        if atom.GetSymbol() in METALS:
            return True

    return False

In [8]:
# ============================================================
# STRUCTURE QUALITY CONTROL
# ============================================================

ALLOWED_ELEMENTS = {
    "C",
    "H",
    "N",
    "O",
    "S",
    "P",
    "F",
    "Cl",
    "Br",
    "I"
}


def structure_passes_qc(mol):

    if mol is None:
        return False, "Invalid molecule"

    # ------------------------------------------
    # Must contain carbon
    # ------------------------------------------

    if not any(
        atom.GetSymbol() == "C"
        for atom in mol.GetAtoms()
    ):
        return False, "No carbon"

    # ------------------------------------------
    # No metals
    # ------------------------------------------

    if contains_metal(mol):
        return False, "Metal-containing"

    # ------------------------------------------
    # Check allowed elements
    # ------------------------------------------

    for atom in mol.GetAtoms():

        element = atom.GetSymbol()

        if element not in ALLOWED_ELEMENTS:

            return (
                False,
                f"Unsupported element: {element}"
            )

    # ------------------------------------------
    # Must be a single molecular fragment
    # ------------------------------------------

    fragments = Chem.GetMolFrags(
        mol,
        asMols=True,
        sanitizeFrags=True
    )

    if len(fragments) != 1:

        return False, "Multi-fragment"

    # ------------------------------------------
    # Reject isotopically labelled structures
    # ------------------------------------------

    for atom in mol.GetAtoms():

        if atom.GetIsotope() != 0:

            return False, "Isotopically labelled"

    return True, "PASS"

In [9]:
# ============================================================
# 3D STRUCTURE GENERATION
# ============================================================

def generate_3d_molecule(
    smiles,
    cid,
    parent_ligand
):

    if not smiles:
        return None, "No SMILES"

    try:

        mol = Chem.MolFromSmiles(
            smiles
        )

        if mol is None:
            return None, "Invalid SMILES"

        # Add hydrogens
        mol = Chem.AddHs(mol)

        # --------------------------------------
        # Generate 3D coordinates
        # --------------------------------------

        params = AllChem.ETKDGv3()

        params.randomSeed = RANDOM_SEED

        result = AllChem.EmbedMolecule(
            mol,
            params
        )

        # Try again if first embedding fails
        if result != 0:

            params.useRandomCoords = True

            result = AllChem.EmbedMolecule(
                mol,
                params
            )

        if result != 0:

            return None, "3D embedding failed"

        # --------------------------------------
        # Geometry optimization
        # --------------------------------------

        optimization = "None"

        try:

            if rdForceFieldHelpers.MMFFHasAllMoleculeParams(
                mol
            ):

                result = (
                    rdForceFieldHelpers
                    .MMFFOptimizeMolecule(mol)
                )

                optimization = "MMFF"

            elif rdForceFieldHelpers.UFFHasAllMoleculeParams(
                mol
            ):

                result = (
                    rdForceFieldHelpers
                    .UFFOptimizeMolecule(mol)
                )

                optimization = "UFF"

        except Exception:

            optimization = "None"

        # --------------------------------------
        # Metadata
        # --------------------------------------

        mol.SetProp(
            "Parent_Ligand",
            str(parent_ligand)
        )

        mol.SetProp(
            "PubChem_CID",
            str(cid)
        )

        mol.SetProp(
            "PubChem_Similarity",
            f">={SIMILARITY_THRESHOLD}% 2D Tanimoto"
        )

        mol.SetProp(
            "3D_Generation",
            "RDKit ETKDGv3"
        )

        mol.SetProp(
            "3D_Optimization",
            optimization
        )

        return mol, "PASS"

    except Exception as e:

        return None, str(e)

In [10]:
# ============================================================
# VERIFY ALL 12 PARENT COMPOUNDS
# ============================================================

parent_check_records = []

for ligand, cid in ligand_cids.items():

    try:

        records = get_properties_for_cids(
            [cid]
        )

        if records:

            data = records[0]

            parent_check_records.append({
                "Ligand": ligand,
                "CID": cid,
                "PubChem_Title": data.get(
                    "Title"
                ),
                "IUPACName": data.get(
                    "IUPACName"
                ),
                "SMILES": data.get(
                    "IsomericSMILES"
                ) or data.get(
                    "CanonicalSMILES"
                ),
                "Formula": data.get(
                    "MolecularFormula"
                )
            })

        else:

            parent_check_records.append({
                "Ligand": ligand,
                "CID": cid,
                "PubChem_Title": "NOT FOUND",
                "IUPACName": "",
                "SMILES": "",
                "Formula": ""
            })

    except Exception as e:

        parent_check_records.append({
            "Ligand": ligand,
            "CID": cid,
            "PubChem_Title": f"ERROR: {e}",
            "IUPACName": "",
            "SMILES": "",
            "Formula": ""
        })

    time.sleep(0.5)


parent_check_df = pd.DataFrame(
    parent_check_records
)

display(parent_check_df)

,Ligand,CID,PubChem_Title,IUPACName,SMILES,Formula
0,Piperazine,4837,Piperazine,piperazine,None,C4H10N2
1,Morphine,5288826,Morphine,"(4R,4aR,7S,7aR,12bS)-3-methyl-2,4,4a,7,7a,13-h...",None,C17H19NO3
2,Azepane,8119,Azepane,azepane,None,C6H13N
3,Azetidine,10422,Azetidine,azetidine,None,C3H7N
4,Imidazole,795,Imidazole,1H-imidazole,None,C3H4N2
5,Pyrazine,9261,Pyrazine,pyrazine,None,C4H4N2
6,Quinoline,7047,Quinoline,quinoline,None,C9H7N
7,Tetrahydrofuran,8028,Tetrahydrofuran,oxolane,None,C4H8O
8,Tetrahydropyran,8894,Tetrahydropyran,oxane,None,C5H10O
9,Dioxolane,12586,"1,3-Dioxolane","1,3-dioxolane",None,C3H6O2


In [11]:
# ============================================================
# LOCKED PARAMETERS — DO NOT CHANGE
# ============================================================

SIMILARITY_THRESHOLD = 90
MAX_CANDIDATES = 1000
RANDOM_SEED = 42

print("SIMILARITY_THRESHOLD =", SIMILARITY_THRESHOLD)
print("MAX_CANDIDATES =", MAX_CANDIDATES)
print("RANDOM_SEED =", RANDOM_SEED)

SIMILARITY_THRESHOLD = 90
MAX_CANDIDATES = 1000
RANDOM_SEED = 42


In [12]:
# ============================================================
# FINAL PROCESSING OF ALL 24 LIGANDS
# ============================================================

all_summary = []

for ligand_number, (
    ligand,
    parent_cid
) in enumerate(
    ligand_cids.items(),
    start=1
):

    print("\n")
    print("=" * 75)
    print(
        f"{ligand_number:02d}/24  {ligand}"
    )
    print(
        f"Parent CID: {parent_cid}"
    )
    print("=" * 75)

    # ------------------------------------------
    # Folders
    # ------------------------------------------

    ligand_folder = (
        LIGAND_DIR /
        f"{ligand_number:02d}_{ligand}"
    )

    raw_folder = (
        ligand_folder / "raw"
    )

    qc_folder = (
        ligand_folder / "QC_clean"
    )

    # ------------------------------------------
    # 1. PubChem similarity search
    # ------------------------------------------

    print("\n[1/7] Searching PubChem...")

    candidate_cids = find_similar_cids(
        parent_cid,
        threshold=SIMILARITY_THRESHOLD,
        max_records=MAX_CANDIDATES
    )

    total_hits = len(candidate_cids)

    print(
        "Total similarity hits:",
        total_hits
    )

    # ------------------------------------------
    # Keep parent ligand
    # ------------------------------------------

    candidate_cids = [
        parent_cid
    ] + [
        cid
        for cid in candidate_cids
        if cid != parent_cid
    ]

    # Remove duplicates
    candidate_cids = list(
        dict.fromkeys(candidate_cids)
    )

    print(
        "Candidates including parent:",
        len(candidate_cids)
    )

    # ------------------------------------------
    # 2. Get compound properties
    # ------------------------------------------

    print(
        "\n[2/7] Downloading compound structures..."
    )

    records = get_properties_for_cids(
        candidate_cids
    )

    df = pd.DataFrame(records)

    print(
        "PubChem records retrieved:",
        len(df)
    )

    # ------------------------------------------
    # Save raw metadata
    # ------------------------------------------

    raw_csv = (
        raw_folder /
        f"{ligand}_raw_candidates.csv"
    )

    df.to_csv(
        raw_csv,
        index=False
    )

    # ------------------------------------------
    # 3. Identify SMILES column
    # ------------------------------------------

    print(
        "\n[3/7] Checking SMILES..."
    )

    smiles_column = None

    for column in [
        "IsomericSMILES",
        "CanonicalSMILES",
        "ConnectivitySMILES"
    ]:

        if column in df.columns:

            smiles_column = column
            break

    if smiles_column is None:

        print(
            "NO SMILES COLUMN FOUND."
        )

        all_summary.append({
            "Ligand": ligand,
            "Parent_CID": parent_cid,
            "Similarity_Hits": total_hits,
            "Candidates": len(candidate_cids),
            "PubChem_Records": len(df),
            "QC_Pass": 0,
            "QC_Fail": len(candidate_cids),
            "3D_Success": 0,
            "3D_Failed": 0
        })

        continue

    print(
        "Using SMILES column:",
        smiles_column
    )

    # ------------------------------------------
    # 4. Structure QC
    # ------------------------------------------

    print(
        "\n[4/7] Performing chemical QC..."
    )

    qc_pass_records = []

    metal_count = 0
    fragment_count = 0
    element_count = 0
    isotope_count = 0
    invalid_count = 0

    for _, row in df.iterrows():

        cid = int(row["CID"])

        smiles = row.get(
            smiles_column
        )

        if pd.isna(smiles):

            invalid_count += 1
            continue

        mol = Chem.MolFromSmiles(
            smiles
        )

        if mol is None:

            invalid_count += 1
            continue

        passes, reason = (
            structure_passes_qc(mol)
        )

        if passes:

            qc_pass_records.append(
                row.to_dict()
            )

        else:

            if reason == "Metal-containing":
                metal_count += 1

            elif reason == "Multi-fragment":
                fragment_count += 1

            elif (
                reason.startswith(
                    "Unsupported element"
                )
            ):
                element_count += 1

            elif reason == "Isotopically labelled":
                isotope_count += 1

            else:
                invalid_count += 1

    qc_df = pd.DataFrame(
        qc_pass_records
    )

    print(
        "QC-passing structures:",
        len(qc_df)
    )

    print(
        "Metal-containing removed:",
        metal_count
    )

    print(
        "Multi-fragment removed:",
        fragment_count
    )

    print(
        "Unsupported element removed:",
        element_count
    )

    print(
        "Isotope-labelled removed:",
        isotope_count
    )

    print(
        "Other invalid removed:",
        invalid_count
    )

    # ------------------------------------------
    # Save QC metadata
    # ------------------------------------------

    qc_csv = (
        qc_folder /
        f"{ligand}_QC_metadata.csv"
    )

    qc_df.to_csv(
        qc_csv,
        index=False
    )

    # ------------------------------------------
    # 5. Generate 3D
    # ------------------------------------------

    print(
        "\n[5/7] Generating 3D structures..."
    )

    final_sdf = (
        qc_folder /
        f"{ligand}_derivatives_3D_QC_clean.sdf"
    )

    writer = SDWriter(
        str(final_sdf)
    )

    success_3d = 0
    failed_3d = 0

    failed_cids = []

    for _, row in qc_df.iterrows():

        cid = int(row["CID"])

        smiles = row.get(
            smiles_column
        )

        mol, status = generate_3d_molecule(
            smiles,
            cid,
            ligand
        )

        if mol is None:

            failed_3d += 1
            failed_cids.append(cid)

            continue

        # --------------------------------------
        # Add PubChem metadata
        # --------------------------------------

        if pd.notna(
            row.get("Title")
        ):

            mol.SetProp(
                "PubChem_Title",
                str(row["Title"])
            )

        if pd.notna(
            row.get("IUPACName")
        ):

            mol.SetProp(
                "IUPAC_Name",
                str(row["IUPACName"])
            )

        if pd.notna(
            row.get("MolecularFormula")
        ):

            mol.SetProp(
                "Molecular_Formula",
                str(
                    row[
                        "MolecularFormula"
                    ]
                )
            )

        if pd.notna(
            row.get("MolecularWeight")
        ):

            mol.SetProp(
                "Molecular_Weight",
                str(
                    row[
                        "MolecularWeight"
                    ]
                )
            )

        writer.write(mol)

        success_3d += 1

    writer.close()

    # ------------------------------------------
    # 6. Verify generated SDF
    # ------------------------------------------

    print(
        "\n[6/7] Verifying SDF..."
    )

    supplier = Chem.SDMolSupplier(
        str(final_sdf),
        removeHs=False
    )

    verified_molecules = [
        mol
        for mol in supplier
        if mol is not None
    ]

    verified_count = len(
        verified_molecules
    )

    # ------------------------------------------
    # 7. Save failed CID list
    # ------------------------------------------

    failed_file = (
        qc_folder /
        f"{ligand}_failed_3D_CIDs.txt"
    )

    with open(
        failed_file,
        "w"
    ) as f:

        for cid in failed_cids:

            f.write(
                f"{cid}\n"
            )

    print(
        "\n[7/7] COMPLETE"
    )

    print(
        "Final 3D structures:",
        verified_count
    )

    print(
        "SDF:",
        final_sdf
    )

    # ------------------------------------------
    # Summary
    # ------------------------------------------

    all_summary.append({

        "Ligand": ligand,

        "Parent_CID": parent_cid,

        "Similarity_Hits":
            total_hits,

        "Candidates_After_Parent":
            len(candidate_cids),

        "PubChem_Records":
            len(df),

        "QC_Pass":
            len(qc_df),

        "Metal_Removed":
            metal_count,

        "MultiFragment_Removed":
            fragment_count,

        "UnsupportedElement_Removed":
            element_count,

        "Isotope_Removed":
            isotope_count,

        "Other_Invalid_Removed":
            invalid_count,

        "3D_Success":
            success_3d,

        "3D_Failed":
            failed_3d,

        "SDF_Verified":
            verified_count,

        "SDF_Path":
            str(final_sdf)
    })

    # Be polite to PubChem
    time.sleep(1)



01/24  Piperazine
Parent CID: 4837

[1/7] Searching PubChem...
Total similarity hits: 312
Candidates including parent: 312

[2/7] Downloading compound structures...
PubChem records retrieved: 312

[3/7] Checking SMILES...
Using SMILES column: ConnectivitySMILES

[4/7] Performing chemical QC...
QC-passing structures: 137
Metal-containing removed: 71
Multi-fragment removed: 69
Unsupported element removed: 35
Isotope-labelled removed: 0
Other invalid removed: 0

[5/7] Generating 3D structures...


[05:48:16] WARNING: not removing hydrogen atom without neighbors
[05:48:16] WARNING: not removing hydrogen atom without neighbors
[05:48:16] WARNING: not removing hydrogen atom without neighbors
[05:48:16] WARNING: not removing hydrogen atom without neighbors
[05:48:16] WARNING: not removing hydrogen atom without neighbors
[05:48:16] WARNING: not removing hydrogen atom without neighbors
[05:48:16] WARNING: not removing hydrogen atom without neighbors
[05:48:16] WARNING: not removing hydrogen atom without neighbors
[05:48:16] WARNING: not removing hydrogen atom without neighbors
[05:48:16] WARNING: not removing hydrogen atom without neighbors
[05:48:16] WARNING: not removing hydrogen atom without neighbors
[05:48:16] WARNING: not removing hydrogen atom without neighbors
[05:48:16] WARNING: not removing hydrogen atom without neighbors
[05:48:16] WARNING: not removing hydrogen atom without neighbors
[05:48:16] WARNING: not removing hydrogen atom without neighbors
[05:48:16] WARNING: not r


[6/7] Verifying SDF...

[7/7] COMPLETE
Final 3D structures: 137
SDF: /content/ICM_Ligand_Derivative_Project_FINAL/01_Ligands/01_Piperazine/QC_clean/Piperazine_derivatives_3D_QC_clean.sdf


02/24  Morphine
Parent CID: 5288826

[1/7] Searching PubChem...
Total similarity hits: 1000
Candidates including parent: 1000

[2/7] Downloading compound structures...
PubChem records retrieved: 1000

[3/7] Checking SMILES...
Using SMILES column: ConnectivitySMILES

[4/7] Performing chemical QC...


[05:48:32] WARNING: not removing hydrogen atom without neighbors


QC-passing structures: 831
Metal-containing removed: 0
Multi-fragment removed: 165
Unsupported element removed: 4
Isotope-labelled removed: 0
Other invalid removed: 0

[5/7] Generating 3D structures...

[6/7] Verifying SDF...

[7/7] COMPLETE
Final 3D structures: 831
SDF: /content/ICM_Ligand_Derivative_Project_FINAL/01_Ligands/02_Morphine/QC_clean/Morphine_derivatives_3D_QC_clean.sdf


03/24  Azepane
Parent CID: 8119

[1/7] Searching PubChem...
Total similarity hits: 81
Candidates including parent: 81

[2/7] Downloading compound structures...
PubChem records retrieved: 81

[3/7] Checking SMILES...
Using SMILES column: ConnectivitySMILES

[4/7] Performing chemical QC...
QC-passing structures: 31
Metal-containing removed: 11
Multi-fragment removed: 34
Unsupported element removed: 5
Isotope-labelled removed: 0
Other invalid removed: 0

[5/7] Generating 3D structures...


...


[6/7] Verifying SDF...

[7/7] COMPLETE
Final 3D structures: 60
SDF: /content/ICM_Ligand_Derivative_Project_FINAL/01_Ligands/22_Triazole/QC_clean/Triazole_derivatives_3D_QC_clean.sdf


23/24  Tetrazole
Parent CID: 67519

[1/7] Searching PubChem...
Total similarity hits: 101
Candidates including parent: 101

[2/7] Downloading compound structures...
PubChem records retrieved: 101

[3/7] Checking SMILES...
Using SMILES column: ConnectivitySMILES

[4/7] Performing chemical QC...
QC-passing structures: 15
Metal-containing removed: 53
Multi-fragment removed: 22
Unsupported element removed: 11
Isotope-labelled removed: 0
Other invalid removed: 0

[5/7] Generating 3D structures...

[6/7] Verifying SDF...

[7/7] COMPLETE
Final 3D structures: 15
SDF: /content/ICM_Ligand_Derivative_Project_FINAL/01_Ligands/23_Tetrazole/QC_clean/Tetrazole_derivatives_3D_QC_clean.sdf


[05:55:49] WARNING: not removing hydrogen atom without neighbors
[05:55:49] WARNING: not removing hydrogen atom without neighbors
[05:55:49] WARNING: not removing hydrogen atom without neighbors
[05:55:49] WARNING: not removing hydrogen atom without neighbors
[05:55:49] WARNING: not removing hydrogen atom without neighbors
[05:55:49] WARNING: not removing hydrogen atom without neighbors
[05:55:49] WARNING: not removing hydrogen atom without neighbors
[05:55:49] WARNING: not removing hydrogen atom without neighbors
[05:55:49] WARNING: not removing hydrogen atom without neighbors
[05:55:49] WARNING: not removing hydrogen atom without neighbors
[05:55:49] WARNING: not removing hydrogen atom without neighbors
[05:55:49] WARNING: not removing hydrogen atom without neighbors
[05:55:49] WARNING: not removing hydrogen atom without neighbors
[05:55:49] WARNING: not removing hydrogen atom without neighbors
[05:55:49] WARNING: not removing hydrogen atom without neighbors
[05:55:49] WARNING: not r



24/24  Indole
Parent CID: 1981

[1/7] Searching PubChem...
Total similarity hits: 1000
Candidates including parent: 1000

[2/7] Downloading compound structures...
PubChem records retrieved: 1000

[3/7] Checking SMILES...
Using SMILES column: ConnectivitySMILES

[4/7] Performing chemical QC...
QC-passing structures: 871
Metal-containing removed: 11
Multi-fragment removed: 94
Unsupported element removed: 24
Isotope-labelled removed: 0
Other invalid removed: 0

[5/7] Generating 3D structures...


[05:56:15] UFFTYPER: Unrecognized charge state for atom: 30
[05:56:22] UFFTYPER: Unrecognized charge state for atom: 7
[05:56:22] UFFTYPER: Unrecognized charge state for atom: 30
[05:58:35] UFFTYPER: Warning: hybridization set to SP3 for atom 23



[6/7] Verifying SDF...

[7/7] COMPLETE
Final 3D structures: 871
SDF: /content/ICM_Ligand_Derivative_Project_FINAL/01_Ligands/24_Indole/QC_clean/Indole_derivatives_3D_QC_clean.sdf


In [13]:
# ============================================================
# FINAL SUMMARY TABLE
# ============================================================

summary_df = pd.DataFrame(
    all_summary
)

display(summary_df)

,Ligand,Parent_CID,Similarity_Hits,Candidates_After_Parent,PubChem_Records,QC_Pass,Metal_Removed,MultiFragment_Removed,UnsupportedElement_Removed,Isotope_Removed,Other_Invalid_Removed,3D_Success,3D_Failed,SDF_Verified,SDF_Path
0,Piperazine,4837,312,312,312,137,71,69,35,0,0,137,0,137,/content/ICM_Ligand_Derivative_Project_FINAL/0...
1,Morphine,5288826,1000,1000,1000,831,0,165,4,0,0,831,0,831,/content/ICM_Ligand_Derivative_Project_FINAL/0...
2,Azepane,8119,81,81,81,31,11,34,5,0,0,31,0,31,/content/ICM_Ligand_Derivative_Project_FINAL/0...
3,Azetidine,10422,65,65,65,32,13,16,4,0,0,32,0,32,/content/ICM_Ligand_Derivative_Project_FINAL/0...
4,Imidazole,795,1000,1000,1000,316,171,410,101,0,2,316,0,316,/content/ICM_Ligand_Derivative_Project_FINAL/0...
5,Pyrazine,9261,200,200,200,37,42,82,39,0,0,36,1,36,/content/ICM_Ligand_Derivative_Project_FINAL/0...
6,Quinoline,7047,1000,1000,1000,830,5,125,40,0,0,828,2,828,/content/ICM_Ligand_Derivative_Project_FINAL/0...
7,Tetrahydrofuran,8028,227,227,227,33,113,41,40,0,0,33,0,33,/content/ICM_Ligand_Derivative_Project_FINAL/0...
8,Tetrahydropyran,8894,156,156,156,36,68,19,33,0,0,36,0,36,/content/ICM_Ligand_Derivative_Project_FINAL/0...
9,Dioxolane,12586,51,51,51,20,16,7,8,0,0,20,0,20,/content/ICM_Ligand_Derivative_Project_FINAL/0...


In [14]:
# ============================================================
# SAVE FINAL SUMMARY
# ============================================================

summary_csv = (
    SUMMARY_DIR /
    "ALL_24_LIGANDS_FINAL_SUMMARY.csv"
)

summary_df.to_csv(
    summary_csv,
    index=False
)

print(
    "Summary saved to:"
)

print(summary_csv)

Summary saved to:
/content/ICM_Ligand_Derivative_Project_FINAL/03_Summary/ALL_24_LIGANDS_FINAL_SUMMARY.csv


In [15]:
# ============================================================
# FINAL SDF QUALITY CHECK
# ============================================================

verification_results = []

for _, row in summary_df.iterrows():

    ligand = row["Ligand"]

    sdf_path = row["SDF_Path"]

    path = Path(sdf_path)

    if not path.exists():

        verification_results.append({
            "Ligand": ligand,
            "SDF_Exists": False,
            "Molecules": 0,
            "All_3D": False
        })

        continue

    supplier = Chem.SDMolSupplier(
        str(path),
        removeHs=False
    )

    molecules = [
        mol
        for mol in supplier
        if mol is not None
    ]

    all_3d = True

    for mol in molecules:

        if mol.GetNumConformers() == 0:

            all_3d = False
            break

    verification_results.append({

        "Ligand": ligand,

        "SDF_Exists": True,

        "Molecules":
            len(molecules),

        "All_3D":
            all_3d
    })


verification_df = pd.DataFrame(
    verification_results
)

display(verification_df)

[05:58:39] Explicit valence for atom # 8 S, 6, is greater than permitted
[05:58:39] ERROR: Could not sanitize molecule ending on line 11926
[05:58:39] ERROR: Explicit valence for atom # 8 S, 6, is greater than permitted


,Ligand,SDF_Exists,Molecules,All_3D
0,Piperazine,True,137,True
1,Morphine,True,831,True
2,Azepane,True,31,True
3,Azetidine,True,32,True
4,Imidazole,True,316,True
5,Pyrazine,True,36,True
6,Quinoline,True,828,True
7,Tetrahydrofuran,True,33,True
8,Tetrahydropyran,True,36,True
9,Dioxolane,True,20,True


In [16]:
# ============================================================
# CREATE FINAL ZIP
# ============================================================

import shutil

zip_path = shutil.make_archive(
    "/content/ICM_Ligand_Derivative_Project_FINAL",
    "zip",
    PROJECT_DIR
)

print("FINAL ZIP:")
print(zip_path)

FINAL ZIP:
/content/ICM_Ligand_Derivative_Project_FINAL.zip


In [17]:
# ============================================================
# DOWNLOAD FINAL PROJECT
# ============================================================

from google.colab import files

files.download(
    "/content/ICM_Ligand_Derivative_Project_FINAL.zip"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>